# Lab 3.12 — Fine-Tuning a Small Encoder
Compare frozen transfer, conservative fine-tuning, and an intentionally aggressive update from the same checkpoint.


In [ ]:
import copy, random, numpy as np, torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
SEED=13; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(1)
d=load_digits(); X=d.data.astype('float32')/16.0; y=d.target.astype('int64'); Xt,Xv,yt,yv=train_test_split(X,y,test_size=0.25,random_state=SEED,stratify=y)
class Enc(torch.nn.Module):
    def __init__(self): super().__init__(); self.net=torch.nn.Sequential(torch.nn.Linear(64,32),torch.nn.ReLU())
    def forward(self,x): return self.net(x)
class M(torch.nn.Module):
    def __init__(self,enc=None,out=2): super().__init__(); self.encoder=enc or Enc(); self.head=torch.nn.Linear(32,out)
    def forward(self,x): return self.head(self.encoder(x))
def train(m,a,b,epochs,lr):
    opt=torch.optim.SGD([p for p in m.parameters() if p.requires_grad],lr=lr); f=torch.nn.CrossEntropyLoss()
    for _ in range(epochs): opt.zero_grad(); z=f(m(a),b); z.backward(); opt.step()
    return m
def acc(m,a,b):
    m.eval();
    with torch.no_grad(): return float((m(a).argmax(1)==b).float().mean())
src=M(out=10); src=train(src,torch.tensor(Xt),torch.tensor(yt),120,0.1)
target=np.isin(yt,[0,6,8,9]).astype('int64'); targetv=torch.tensor(np.isin(yv,[0,6,8,9]).astype('int64')); idx=np.arange(len(yt)); small,_=train_test_split(idx,train_size=40,random_state=SEED,stratify=target); a=torch.tensor(Xt[small]); b=torch.tensor(target[small]); val=torch.tensor(Xv)
enc=copy.deepcopy(src.encoder); [setattr(p,'requires_grad',False) for p in enc.parameters()]; torch.manual_seed(SEED); frozen=train(M(enc),a,b,200,0.2)
def vec(m): return torch.cat([p.detach().flatten() for p in m.encoder.parameters()])
base=vec(frozen).clone(); fine=copy.deepcopy(frozen); [setattr(p,'requires_grad',True) for p in fine.encoder.parameters()]; fine=train(fine,a,b,80,0.01)
aggressive=copy.deepcopy(frozen); [setattr(p,'requires_grad',True) for p in aggressive.encoder.parameters()]; aggressive=train(aggressive,a,b,80,1.0)
fm=float(torch.linalg.vector_norm(vec(fine)-base)); am=float(torch.linalg.vector_norm(vec(aggressive)-base)); fa=acc(frozen,val,targetv); fta=acc(fine,val,targetv); aa=acc(aggressive,val,targetv)
print(f'frozen_accuracy={fa:.3f}'); print(f'fine_tuned_accuracy={fta:.3f} movement={fm:.3f}'); print(f'aggressive_accuracy={aa:.3f} movement={am:.3f}')
assert fta>0.75 and am>fm*2
